## Descrição
Esse notebook carrega os dados da camada bronze e realiza a limpeza, removendo registros com valores negativos e de ilhas de descarga. Depois salva os dados na camada silver.

In [0]:
spark.sql("USE CATALOG mvp")
spark.sql("USE SCHEMA silver")

DataFrame[]

In [0]:
from pyspark.sql.functions import col

df = spark.table("mvp.bronze.ind_prod_ilha")
print("Antes da limpeza de dados: %d", df.count())

#Elimina os registros com valores negativos
df_filtered = df.filter(
    (col('TempoProducaoReal') >= 0) &
    (col('TempoProducaoPlan') >= 0) &
    (col('QuantProducaoReal') >= 0) &
    (col('QuantProducaoPlan') >= 0) &
    (col('QuantProducaoRuim') >= 0) &
    (col('QuantProducaoBoa') >= 0)
)

print("Depois da limpeza de dados: %d", df_filtered.count())

# muda o tipo dos campos numéricos para double
df_filtered = (
    df_filtered
    .withColumn("TempoProducaoReal", col("TempoProducaoReal").cast("double"))
    .withColumn("TempoProducaoPlan", col("TempoProducaoPlan").cast("double"))
    .withColumn("QuantProducaoReal", col("QuantProducaoReal").cast("double"))
    .withColumn("QuantProducaoPlan", col("QuantProducaoPlan").cast("double"))
    .withColumn("QuantProducaoRuim", col("QuantProducaoRuim").cast("double"))
    .withColumn("QuantProducaoBoa", col("QuantProducaoBoa").cast("double"))
)

# Salva a tabela e o schema, devido à mudança do tipo
df_filtered.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ind_prod_ilha")

Antes da limpeza de dados: %d 5760
Depois da limpeza de dados: %d 5650


In [0]:
## atualiza os comentários dos campos
spark.sql("""
  COMMENT ON COLUMN ind_prod_ilha.TempoProducaoReal IS 'Tempo de produção real da ilha'
""")
spark.sql("""
  COMMENT ON COLUMN ind_prod_ilha.TempoProducaoPlan IS 'Tempo de produção planejado da ilha'
""")
spark.sql("""
  COMMENT ON COLUMN ind_prod_ilha.QuantProducaoReal IS 'Quantidade de produção real da ilha'
""")
spark.sql("""
  COMMENT ON COLUMN ind_prod_ilha.QuantProducaoPlan IS 'Quantidade de produção planejada da ilha'
""")
spark.sql("""
  COMMENT ON COLUMN ind_prod_ilha.QuantProducaoRuim IS 'Quantidade de produção ruim da ilha'
""")
spark.sql("""
  COMMENT ON COLUMN ind_prod_ilha.QuantProducaoBoa IS 'Quantidade de produção boa da ilha'
""")

DataFrame[]

In [0]:
%sql
-- Apaga os registros da tabela ind_prod_ilha que são de ilhas de descarga, pois esse projeto é focado nas ilhas de carga
DELETE FROM mvp.silver.ind_prod_ilha
WHERE IdIlha IN (
    SELECT IdIlha FROM mvp.bronze.ilhas WHERE tipo = 'D'
)

num_affected_rows
734


In [0]:
%sql
SELECT *
FROM ind_prod_ilha 
LIMIT 10

IdBase,Data,IdIlha,TempoProducaoReal,TempoProducaoPlan,QuantProducaoReal,QuantProducaoPlan,QuantProducaoRuim,QuantProducaoBoa
1,2025-11-01,50121,26513.0,57600.0,383117.0,1740000.0,0.0,65.0
1,2025-11-01,501213,9483.0,57600.0,122347.0,870000.0,10.0,13.0
1,2025-11-01,501219,6319.0,57600.0,82999.0,870000.0,0.0,16.0
1,2025-11-01,50122,22583.0,57600.0,340031.0,1740000.0,0.0,58.0
1,2025-11-01,50123,25494.0,57600.0,444020.0,1740000.0,0.0,91.0
1,2025-11-01,50124,25462.0,57600.0,441043.0,1740000.0,0.0,91.0
1,2025-11-01,50125,29855.0,57600.0,463091.0,1740000.0,0.0,96.0
1,2025-11-01,50126,23201.0,57600.0,392061.0,3060000.0,0.0,78.0
1,2025-11-03,50121,36095.0,57600.0,521125.0,1740000.0,1.0,96.0
1,2025-11-03,501213,4417.0,57600.0,60002.0,870000.0,0.0,12.0


In [0]:
# Cria as tabelas bases e ilhas no schema silver
df = spark.table("mvp.bronze.bases")
display(df.limit(2))
df.write.mode("overwrite").saveAsTable("bases")
df = spark.table("mvp.bronze.ilhas")
display(df.limit(2))
df.write.mode("overwrite").saveAsTable("ilhas")

IdBase,Nome
1,Base 1
5,Base 5


IdBase,IdIlha,Nome,Tipo
1,50121,Ilha 1,T
1,50122,Ilha 2,T
